# 00. Setup and Data Audit

Start here. This notebook checks whether the exported data is complete enough to support analysis before calculating any headline metric.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent

sys.path.insert(0, str(WORKSPACE / "src"))

from venture_ds import (
    load_exports,
    load_optional_export,
    safe_rate,
    wilson_interval,
    ordered_session_funnel,
    add_sample_warning,
)

exports = load_exports(WORKSPACE / "data" / "raw")
events = exports["events"]
sessions = exports["sessions"]
feedback = exports["feedback"]
inventory = exports["inventory"]

exports["_files"]

,dataset,path,found
0,events,C:\Users\ppmal\Documents\Projects\venture-prod...,True
1,sessions,C:\Users\ppmal\Documents\Projects\venture-prod...,True
2,feedback,C:\Users\ppmal\Documents\Projects\venture-prod...,True
3,inventory,C:\Users\ppmal\Documents\Projects\venture-prod...,True


In [2]:
datasets = {
    "events": events,
    "sessions": sessions,
    "feedback": feedback,
    "inventory": inventory,
}

audit = []

for name, frame in datasets.items():
    if frame.empty:
        duplicate_rows = 0
        missing_cells = 0
    else:
        duplicate_frame = frame.copy()

        # Convert dictionaries and lists into comparable strings.
        for column in duplicate_frame.columns:
            duplicate_frame[column] = duplicate_frame[column].map(
                lambda value: json.dumps(
                    value,
                    sort_keys=True,
                    default=str,
                )
                if isinstance(value, (dict, list, tuple, set))
                else value
            )

        duplicate_rows = int(
            duplicate_frame.duplicated().sum()
        )

        missing_cells = int(
            frame.isna().sum().sum()
        )

    audit.append({
        "dataset": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "duplicate_rows": duplicate_rows,
        "missing_cells": missing_cells,
    })

audit_table = pd.DataFrame(audit)

audit_table

,dataset,rows,columns,duplicate_rows,missing_cells
0,events,26,15,0,135
1,sessions,3,15,0,5
2,feedback,0,1,0,0
3,inventory,28,9,0,49


In [3]:
if not events.empty:
    event_audit = (
        events.groupby("event_name", dropna=False)
        .agg(
            rows=("event_name", "size"),
            sessions=("session_id", "nunique"),
            visitors=("visitor_id", "nunique"),
            first_seen=("created_at", "min"),
            last_seen=("created_at", "max"),
        )
        .sort_values("rows", ascending=False)
    )
    display(event_audit)
else:
    print("No events export found.")

,rows,sessions,visitors,first_seen,last_seen
event_name,,,,,
page_view,14,2,1,2026-07-20 19:16:00.793996+00:00,2026-07-20 20:21:02.875059+00:00
feedback_opened,2,1,0,2026-07-20 19:22:31.320463+00:00,2026-07-20 19:22:35.121199+00:00
outing_builder_opened,1,1,1,2026-07-20 20:19:45.273380+00:00,2026-07-20 20:19:45.273380+00:00
experiment_exposure,1,1,1,2026-07-20 20:20:20.658082+00:00,2026-07-20 20:20:20.658082+00:00
outing_generated,1,1,1,2026-07-20 20:20:19.456036+00:00,2026-07-20 20:20:19.456036+00:00
outing_result_viewed,1,1,1,2026-07-20 20:20:19.831007+00:00,2026-07-20 20:20:19.831007+00:00
outing_shared,1,1,1,2026-07-20 20:20:32.612473+00:00,2026-07-20 20:20:32.612473+00:00
outing_saved,1,1,1,2026-07-20 20:20:27.521822+00:00,2026-07-20 20:20:27.521822+00:00
outing_stop_replaced,1,1,1,2026-07-20 20:20:44.504528+00:00,2026-07-20 20:20:44.504528+00:00


In [4]:
quality_checks = {}

if not events.empty:
    quality_checks = {
        "missing_session_id": int(events["session_id"].isna().sum()),
        "missing_visitor_id": int(events["visitor_id"].isna().sum()),
        "duplicate_event_id": int(events["event_id"].duplicated().sum())
            if "event_id" in events.columns else None,
        "invalid_event_version": int((events["event_version"] != 2).sum())
            if "event_version" in events.columns else None,
        "missing_server_time": int(events["created_at"].isna().sum()),
    }

pd.Series(quality_checks, name="count")

missing_session_id        0
missing_visitor_id       12
duplicate_event_id        0
invalid_event_version    12
missing_server_time       0
Name: count, dtype: int64

## Interpretation

Do not continue to strong conclusions when identifiers, timestamps, or event names are materially incomplete. Data quality is part of the analysis, not a chore that happens before it.